# Reuse the exact one-block dataset
Copy `fineTuningMagiCharSimHeads/training_data.pt` without rebuilding triplets or pairs.
This keeps the split, supervision exclusions, hard-negative mix, and held-out data
identical. Rebuild the updated one-block data first if its version check fails.

The dataset was already copied when this folder was created. Run this notebook
only to verify or refresh that copy, then run training and testing.


In [1]:
import torch
import hashlib, shutil
from pathlib import Path
from collections import Counter

ROOT = Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi/fineTuningMagiCharSimHeads2')
SOURCE_FILE = ROOT.parent / 'fineTuningMagiCharSimHeads' / 'training_data.pt'
DATA_FILE = ROOT / 'training_data.pt'

def file_hash(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()

data = torch.load(SOURCE_FILE, map_location='cpu', weights_only=True)
assert data['version'] == 2, 'Rebuild data in the updated one-block notebook first.'
assert data['settings']['excluded_identities'] == ['NARRATION', 'UNKNOWN']
for split, key in [('train', 'train_triplets'), ('validation', 'val_triplets')]:
    rows = data[key]
    assert rows
    counts = Counter(data['appearances'][t['anchor_idx']]['sequence_idx'] for t in rows)
    print(split, '| triplets:', len(rows))
    for i in data['splits'][split]:
        print(data['sequences'][i]['id'], '| triplets:', counts[i], '| share:', f'{counts[i] / len(rows):.1%}')
    if split == 'train':
        assert max(counts.values()) / len(rows) <= data['settings']['max_sequence_share']

# Copy bytes instead of resaving tensors to preserve the exact dataset hash.
source_hash = file_hash(SOURCE_FILE)
shutil.copyfile(SOURCE_FILE, DATA_FILE)
assert file_hash(SOURCE_FILE) == file_hash(DATA_FILE) == source_hash
print('Identical dataset SHA256:', source_hash)
print('Validation pairs:', len(data['val_pairs']), '| test pairs:', len(data['test_pairs']))
print('Saved:', DATA_FILE)


train | triplets: 765
seq_b8b5563330fdc89d | triplets: 22 | share: 2.9%
seq_e8b4f75454f9491f | triplets: 14 | share: 1.8%
seq_5f9ee35846b7e635 | triplets: 8 | share: 1.0%
seq_b9b5a9ad9a64ced9 | triplets: 14 | share: 1.8%
seq_84541a1f4b159018 | triplets: 37 | share: 4.8%
seq_d749b853cf795b6a | triplets: 15 | share: 2.0%
seq_2032620aa4e4ac7f | triplets: 4 | share: 0.5%
seq_b136f32c2233b97c | triplets: 0 | share: 0.0%
seq_d8afceb0e8f9122d | triplets: 8 | share: 1.0%
seq_96839890d69d7698 | triplets: 6 | share: 0.8%
seq_45c9871f52e99417 | triplets: 22 | share: 2.9%
seq_f7f9d9dbf5b05cb5 | triplets: 29 | share: 3.8%
seq_da71f7a842c91090 | triplets: 21 | share: 2.7%
seq_952f154fb1505883 | triplets: 10 | share: 1.3%
seq_a3bd36201a0cd61b | triplets: 0 | share: 0.0%
seq_de542b63e6b19bc2 | triplets: 2 | share: 0.3%
seq_71033fec4de494fc | triplets: 28 | share: 3.7%
seq_d7af6c64ebc22bbe | triplets: 3 | share: 0.4%
seq_fa2eee6269a5dad3 | triplets: 22 | share: 2.9%
seq_06ec0acde37dffa4 | triplets: 7 |